Paso 8 (opcional), muestreo. No es obligatorio pero queria ver si de verdad se nota
lo que dice la teoria del sistematico con k = periodo del ciclo (como el ejemplo de la
guia con k=24 en datos por hora, aqui en minutos el ciclo diario es k=1440).

Trabajo sobre `consumo_limpio.csv` (Paso 2), nada mas filas con medicion, porque muestrear
huecos no tiene sentido (ya lo hicimos asi en distribuciones.py).

In [1]:
from pathlib import Path
import polars as pl
import matplotlib.pyplot as plt

DATA_DIR = Path.cwd().parent / "data" / "data-processed" / "consumo_limpio.csv"

df = pl.read_csv(str(DATA_DIR), try_parse_dates=True)
df = df.filter(~pl.col("medicion_faltante"))
df.height

2049568

Voy a usar `global_active_power` (kW) nomas, es la mas interpretable y la que ya
usamos en el EDA para los picos. Primero la poblacion completa, para comparar contra.

In [2]:
VARIABLE = "global_active_power"
SEMILLA = 2026

media_pob = df[VARIABLE].mean()
std_pob = df[VARIABLE].std()
print(f"poblacion: n={df.height:,}  media={media_pob:.3f} kW  std={std_pob:.3f} kW")

poblacion: n=2,049,568  media=1.092 kW  std=1.057 kW


## Aleatorio simple

Sin reemplazo, ~1% de los minutos. Deberia parecerse mucho a la poblacion.

In [3]:
aleatorio = df.sample(fraction=0.01, seed=SEMILLA)
print(f"n={aleatorio.height:,}  media={aleatorio[VARIABLE].mean():.3f} kW  std={aleatorio[VARIABLE].std():.3f} kW")
# ok esto se ve razonable, la media casi no cambia

n=20,495  media=1.102 kW  std=1.058 kW


## Sistematico

Aqui es donde quiero probar el ejemplo de la guia. Un dia tiene 1440 minutos, entonces
si agarro cada 1440 posiciones SIEMPRE caigo en la misma hora del dia. Eso deberia
sesgar la media hacia lo que pase a esa hora en especifico (por el perfil diario que
ya vimos en el EDA: hay horas pico y horas valle, no es plano).

Para comparar, pruebo tambien con k=1427 (es primo, no es multiplo de 60 ni de 1440,
entonces no deberia coincidir con ningun ciclo del dataset).

In [ ]:
K_SESGADO = 1440   # = un dia en minutos
K_SIN_SESGO = 1427 # primo cercano, no coincide con el ciclo diario ni horario

sistematico_sesgado = df.gather_every(K_SESGADO)
sistematico_ok = df.gather_every(K_SIN_SESGO)

print(f"k={K_SESGADO}: n={sistematico_sesgado.height:,}  media={sistematico_sesgado[VARIABLE].mean():.3f} kW")
print(f"k={K_SIN_SESGO}: n={sistematico_ok.height:,}  media={sistematico_ok[VARIABLE].mean():.3f} kW")

k=1440: n=1,424  media=1.382 kW
k=1427: n=1,437  media=1.078 kW


k=1440 sale bastante mas alto que la poblacion (como 15-16% mas),
exactamente lo que decia la guia que iba a pasar. k=1427 esta mucho mas cerca.

[ `gather_every` sin offset siempre arranca en la fila 0, que es 2006-12-16 17:24.
no se si eso importa mucho con 2 millones de filas pero en teoria deberia probar con
distintos puntos de arranque para estar segura de que no es coincidencia del punto
de inicio y no del periodo. lo dejo como nota, en el script final si lo hice con un
offset random. ]

In [ ]:
# vemos a que hora del dia esta cayendo siempre el sistematico sesgado
sistematico_sesgado.select(pl.col("datetime").dt.hour().value_counts())

datetime
struct[2]
"{10,184}"
"{17,192}"
"{7,228}"
"{22,34}"
"{12,64}"
"{13,59}"
"{8,514}"
"{21,149}"


## Estratificado por mes

Como ya sabemos que hay estacionalidad fuerte (invierno vs verano, visto en el EDA),
tiene sentido estratificar por mes para no perder esa variabilidad. La idea es que cada
mes aporte la misma fraccion de muestra, no una fraccion fija del total.

In [6]:
df_mes = df.with_columns(pl.col("datetime").dt.strftime("%Y-%m").alias("mes"))

# shuffle dentro de cada mes y nos quedamos con el primer 1% de cada uno
estratificado = (
    df_mes
    .filter(pl.int_range(pl.len()).shuffle(seed=SEMILLA).over("mes") < (pl.len() * 0.01).over("mes"))
    .drop("mes")
)
print(f"n={estratificado.height:,}  media={estratificado[VARIABLE].mean():.3f} kW  std={estratificado[VARIABLE].std():.3f} kW")

n=20,516  media=1.086 kW  std=1.053 kW


## Aleatorio con reemplazo (bootstrap)

la clase mencionó "aleatorio sin reemplazo" y "aleatorio con reemplazo" como las dos
variantes del muestreo aleatorio simple, entonces faltaba probar la segunda. la
diferencia es que aqui un mismo minuto puede salir mas de una vez.

con una fraccion chica (1%) no deberia notarse mucho, porque la probabilidad de que un
minuto se repita dentro de la misma muestra es baja. se nota mas si la muestra es casi
del tamano de la poblacion.

In [7]:
bootstrap = df.sample(fraction=0.01, seed=SEMILLA, with_replacement=True)
pct_repetidos = bootstrap["datetime"].is_duplicated().mean() * 100
print(f"n={bootstrap.height:,}  media={bootstrap[VARIABLE].mean():.3f} kW  std={bootstrap[VARIABLE].std():.3f} kW")
print(f"% de minutos repetidos: {pct_repetidos:.2f} %")

n=20,495  media=1.101 kW  std=1.057 kW
% de minutos repetidos: 0.87 %


efectivamente casi no cambia nada vs el aleatorio simple (media casi igual, un
poquito menos de 1% de filas repetidas). tiene sentido: para que el bootstrap aporte
algo distinto habria que compararlo en el caso de uso real, que es estimar la
variabilidad de un estadistico repitiendo el muestreo muchas veces (por ejemplo, la
desviacion estandar de la media entre 1000 muestras bootstrap), no solo sacar una
muestra una vez. aqui solo lo dejo para comparar contra el aleatorio sin reemplazo,
como pide la guia.

## Comparando todo junto

Tabla resumen, el mismo formato que trae la guia (poblacion / aleatorio / sistematico /
estratificado).

In [8]:
resumen = pl.DataFrame([
    {"muestra": "poblacion", "n": df.height, "media_kw": media_pob, "std_kw": std_pob},
    {"muestra": "aleatorio simple", "n": aleatorio.height, "media_kw": aleatorio[VARIABLE].mean(), "std_kw": aleatorio[VARIABLE].std()},
    {"muestra": "aleatorio con reemplazo", "n": bootstrap.height, "media_kw": bootstrap[VARIABLE].mean(), "std_kw": bootstrap[VARIABLE].std()},
    {"muestra": f"sistematico (k={K_SESGADO})", "n": sistematico_sesgado.height, "media_kw": sistematico_sesgado[VARIABLE].mean(), "std_kw": sistematico_sesgado[VARIABLE].std()},
    {"muestra": f"sistematico (k={K_SIN_SESGO})", "n": sistematico_ok.height, "media_kw": sistematico_ok[VARIABLE].mean(), "std_kw": sistematico_ok[VARIABLE].std()},
    {"muestra": "estratificado por mes", "n": estratificado.height, "media_kw": estratificado[VARIABLE].mean(), "std_kw": estratificado[VARIABLE].std()},
]).with_columns(pl.col("media_kw", "std_kw").round(3))
resumen

muestra,n,media_kw,std_kw
str,i64,f64,f64
"""poblacion""",2049568,1.092,1.057
"""aleatorio simple""",20495,1.102,1.058
"""aleatorio con reemplazo""",20495,1.101,1.057
"""sistematico (k=1440)""",1424,1.382,0.989
"""sistematico (k=1427)""",1437,1.078,1.064
"""estratificado por mes""",20516,1.086,1.053


conclusion: el sistematico con k=1440 es justo el caso que 
sesga fuerte porque el periodo de muestreo coincide con el ciclo diario. los demas
(aleatorio, sistematico con k que no coincide, estratificado) se parecen bastante a la
poblacion. 